<a href="https://colab.research.google.com/github/samreenfathima18/guvi-project4/blob/main/imdb_project4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas scikit-learn streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 87.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 99.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [2]:
import string
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def load_data():
    # Replace with your actual CSV file path
    # Sample data matching your dataset structure if file is missing
    try:
        df = pd.read_csv("/content/drive/MyDrive/imdb2024_movies.csv")
    except FileNotFoundError:
        data = {
            "Movie Name": ["The Substance", "The Life of Chuck", "Bone Lake", "Anora", "Eden"],
            "Storyline": [
                "A fading celebrity takes a black-market drug: a cell-replicating substance that temporarily creates a younger, better version of herself.",
                "A life-affirming, genre-bending story about three separate chapters in the life of an ordinary man named Chuck.",
                "A couple's vacation at a secluded estate is upended when they discover they are not alone.",
                "A young stripper from Brooklyn meets and impulsively marries the son of a Russian oligarch.",
                "Based on a factual account of a group of eccentric outsiders who leave society to build their own paradise."
            ]
        }
        df = pd.DataFrame(data)

    # Clean missing values
    df = df.dropna(subset=["Movie Name", "Storyline"]).reset_index(drop=True)
    return df

df = load_data()

# --- 3. TEXT PREPROCESSING ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Pre-calculate cleaned storylines for the dataset
df["Cleaned_Storyline"] = df["Storyline"].apply(clean_text)

# --- 4. NLP VECTORIZATION & SIMILARITY ---
# Using English stop words built into scikit-learn
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["Cleaned_Storyline"])

def get_recommendations(user_query, tfidf_vectorizer, tfidf_mat, dataframe, top_n=5):
    # Clean and vectorize user input
    cleaned_query = clean_text(user_query)
    query_vector = tfidf_vectorizer.transform([cleaned_query])

    # Calculate similarity scores
    similarity_scores = cosine_similarity(query_vector, tfidf_mat).flatten()

    # Get top N indices sorted by highest score
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    # Filter out items with zero similarity if user enters completely random text
    results = []
    for idx in top_indices:
        if similarity_scores[idx] > 0:
            results.append({
                "Movie Name": dataframe.iloc[idx]["Movie Name"],
                "Storyline": dataframe.iloc[idx]["Storyline"],
                "Match Score": f"{similarity_scores[idx] * 100:.1f}%"
            })

    return results



In [8]:
%%writefile app.py
import streamlit as st
import pandas as pd
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. PAGE CONFIGURATION
st.set_page_config(page_title="IMDb Movie Recommender", layout="wide", page_icon="🎬")

# 2. DATA LOADING & CACHING
@st.cache_data
def load_data():
    try:
       df = pd.read_csv("/content/drive/MyDrive/imdb2024_movies.csv")
    except FileNotFoundError:
        # Fallback dataset if your CSV isn't uploaded yet
        data = {
            "Movie Name": ["The Substance", "The Life of Chuck", "Bone Lake", "Anora", "Eden"],
            "Storyline": [
                "A fading celebrity takes a black-market drug: a cell-replicating substance that temporarily creates a younger, better version of herself.",
                "A life-affirming, genre-bending story about three separate chapters in the life of an ordinary man named Chuck.",
                "A couple's vacation at a secluded estate is upended when they discover they are not alone.",
                "A young stripper from Brooklyn meets and impulsively marries the son of a Russian oligarch.",
                "Based on a factual account of a group of eccentric outsiders who leave society to build their own paradise."
            ]
        }
        df = pd.DataFrame(data)

    df = df.dropna(subset=["Movie Name", "Storyline"]).reset_index(drop=True)
    return df

df = load_data()

# 3. TEXT PREPROCESSING
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["Cleaned_Storyline"] = df["Storyline"].apply(clean_text)

# 4. NLP VECTORIZATION & SIMILARITY
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["Cleaned_Storyline"])

def get_recommendations(user_query, tfidf_vectorizer, tfidf_mat, dataframe, top_n=5):
    cleaned_query = clean_text(user_query)
    query_vector = tfidf_vectorizer.transform([cleaned_query])
    similarity_scores = cosine_similarity(query_vector, tfidf_mat).flatten()
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    results = []
    for idx in top_indices:
        if similarity_scores[idx] > 0:
            results.append({
                "Movie Name": dataframe.iloc[idx]["Movie Name"],
                "Storyline": dataframe.iloc[idx]["Storyline"],
                "Match Score": f"{similarity_scores[idx] * 100:.1f}%"
            })
    return results

# 5. STREAMLIT USER INTERFACE
st.title("🎬 IMDb Movie Recommendation System")
st.markdown("Discover 2024 movies by entering a storyline description or plot theme below.")

user_input = st.text_area(
    "Enter a custom storyline or plot concept:",
    placeholder="e.g., A suspenseful vacation at a hidden house where secrets are revealed..."
)

if st.button("Find Similar Movies", type="primary"):
    if user_input.strip() == "":
        st.warning("Please enter a storyline description first.")
    else:
        with st.spinner("Analyzing text similarity..."):
            recommendations = get_recommendations(user_input, tfidf, tfidf_matrix, df)

        if recommendations:
            st.subheader("Top 5 Recommended Movies")
            for i, movie in enumerate(recommendations, 1):
                col1, col2 = st.columns([4, 1])
                with col1:
                    st.markdown(f"### {i}. {movie['Movie Name']}")
                    st.write(movie['Storyline'])
                with col2:
                    st.metric(label="Match Confidence", value=movie['Match Score'])
                st.divider()
        else:
            st.info("No close matches found. Try using different keywords.")


Overwriting app.py


In [9]:
!npm install -g localtunnel -q

⠙⠹⠸⠼⠴
changed 22 packages in 804ms
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [10]:
!streamlit run app.py & npx localtunnel --port 8501


⠙⠹

⠸⠼⠴your url is: https://major-teeth-fry.loca.lt
2026-11-12 15:37:13.903 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.238.26.141:8501

  Stopping...
^C
